In [ ]:
!pip install venn-abers
!pip install mapie==0.8.6

In [2]:
import pandas as pd
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from venn_abers import VennAbers

from mapie.regression import MapieRegressor
from mapie.metrics import regression_coverage_score, regression_mean_width_score
from mapie.subsample import Subsample

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, mean_absolute_error, r2_score, roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor

from scipy import stats
from scipy.stats import ttest_1samp, ttest_ind, bootstrap

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

## Global Configuration

In [3]:
RANDOM_STATE = 21
USE_PREDICTED_STAGE1 = False
STAGE1_MODEL_TYPE = "XGBoost"
K_BINS = 3          
N_BOOT = 1000       

#   Alpha sweep + per-alpha beta ranges
#   beta_max  : upper bound of the beta grid
#   beta_sel_cap : highest beta considered for optimal-beta selection
#                  (keeps intervals from inflating unnecessarily)
ALPHAS_CONFIG = {
    0.3: {"beta_max": 1.5, "beta_sel_cap": 1.0},
    0.2: {"beta_max": 1.5, "beta_sel_cap": 1.0},
    0.1: {"beta_max": 1.5, "beta_sel_cap": 3.0},
    0.05: {"beta_max": 3.0, "beta_sel_cap": 3.0},
}

# Conformal Methods
STRATEGIES = {
    # "naive": {"method": "naive"},
    "split": {'method': 'base'},
    # "cv_plus": {"method": "plus", "cv": 10},
    # "jackknife_plus_ab": {"method": "plus", "cv": Subsample(n_resamplings=50)},
}

# XGBoost Parameters
XGBOOST_BINARY_PARAMS = {
    "eval_metric": "auc",
    "objective": "binary:logistic",
    'sampling_method': 'gradient_based',
    'alpha': 0.1,
    'lambda': 1,
    'learning_rate': 0.1,
    'max_depth': 7,
    'tree_method': 'hist',
    'device': "cuda",
}

XGBOOST_REGRESSION_PARAMS = {
    'alpha': 0.1,
    'lambda': 1,
    'learning_rate': 0.1,
    'max_depth': 7,
    'n_estimators': 700,
    "eval_metric": 'rmse',
    'objective': 'reg:squarederror',
    'sampling_method': 'gradient_based',
    'tree_method': 'hist',
    'device': "cuda"
}

## Data Loading and Preprocessing

In [4]:
path = 'data/data_1Y.csv'
data = pd.read_csv(path)

normalized_pctg_change = data['normalized_percent_change']
data.drop(columns=['normalized_percent_change'], inplace=True)

xgboost_df = data.copy()
demographic_vars = ['gender_source_value', 'race_source_value', 'ethnicity_source_value']
xgboost_df = pd.get_dummies(xgboost_df, columns=demographic_vars)

scaler = MinMaxScaler()
numeric_vars = ['mean_led_per_visit', 'age', 'length_of_stay',
                'days_since_last_visit', 'days_to_diagnosis']
for var in numeric_vars:
    xgboost_df[var] = scaler.fit_transform(xgboost_df[[var]])

prediction_to_last = xgboost_df.pop('prediction')
xgboost_df['prediction'] = prediction_to_last

X          = xgboost_df.iloc[:, :-1]
y_binary   = xgboost_df.iloc[:, -1]
y_continuous = normalized_pctg_change

# Stage 1: Binary Classification with Venn-Abers
Get VA uncertainty estimates for ALL patients in Stage 1 binary prediction. This will inform Stage 2 interval lengths

In [5]:
X_train_stage1, X_test_stage1, y_train_stage1, y_test_stage1 = train_test_split(
    X, y_binary, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Stage 1 | Train: {len(X_train_stage1)} | VA Cal: {len(X_test_stage1)}")

if STAGE1_MODEL_TYPE == "XGBoost":
    d_train = xgb.DMatrix(X_train_stage1, label=y_train_stage1)
    d_test  = xgb.DMatrix(X_test_stage1,  label=y_test_stage1)
    d_all   = xgb.DMatrix(X)
    model_stage1 = xgb.train(
        XGBOOST_BINARY_PARAMS, d_train, num_boost_round=700,
        evals=[(d_test, "validation")], verbose_eval=100, early_stopping_rounds=10
    )
    p_test_uncalibrated = model_stage1.predict(d_test,  iteration_range=(0, model_stage1.best_iteration + 1))
    p_all_uncalibrated  = model_stage1.predict(d_all,   iteration_range=(0, model_stage1.best_iteration + 1))
else:
    model_stage1 = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model_stage1.fit(X_train_stage1, y_train_stage1)
    p_test_uncalibrated = model_stage1.predict_proba(X_test_stage1)[:, 1]
    p_all_uncalibrated  = model_stage1.predict_proba(X)[:, 1]

# Venn-Abers calibration
p_test_2d = np.column_stack([1 - p_test_uncalibrated, p_test_uncalibrated])
va = VennAbers()
va.fit(p_test_2d, y_test_stage1.values)

p_all_2d = np.column_stack([1 - p_all_uncalibrated, p_all_uncalibrated])
p_lower_all_2d, p_upper_all_2d = va.predict_proba(p_all_2d)
p_lower_all      = p_lower_all_2d[:, 1]
p_upper_all      = p_upper_all_2d[:, 1]
va_uncertainty_all = p_upper_all - p_lower_all

print(f"VA Uncertainty | Mean={va_uncertainty_all.mean():.4f}  "
      f"Median={np.median(va_uncertainty_all):.4f}  Max={va_uncertainty_all.max():.4f}")

Stage 1 | Train: 4884 | VA Cal: 1221
[0]	validation-auc:0.86294
[100]	validation-auc:0.95154
[200]	validation-auc:0.96313
[241]	validation-auc:0.96421
VA Uncertainty | Mean=0.0069  Median=0.0003  Max=0.1537


## Stage 1 Uncertainty Validation
Before passing the Stage 1 predictions to the Stage 2 regression model, we validate that the VA interval length is a reliable indicator of epistemic uncertainty.

In [6]:
fpr, tpr, thresholds = roc_curve(y_test_stage1, p_test_uncalibrated)
optimal_idx       = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
y_hat_stage1      = (p_all_uncalibrated > optimal_threshold).astype(int)
errors            = (y_binary != y_hat_stage1).astype(int)

error_auc = roc_auc_score(errors, va_uncertainty_all)
va_wrong  = va_uncertainty_all[errors == 1]
va_correct = va_uncertainty_all[errors == 0]
t_stat, p_val = ttest_ind(va_wrong, va_correct, equal_var=False)

print(f"Optimal Threshold (Youden): {optimal_threshold:.4f}")
print(f"Error Prediction AUROC:     {error_auc:.3f}")
print(f"Mean VA (Errors):   {va_wrong.mean():.4f}")
print(f"Mean VA (Correct):  {va_correct.mean():.4f}")
print(f"P-value:            {p_val:.2e}")

# Outlier analysis
threshold_95      = np.percentile(va_uncertainty_all, 95)
outlier_mask      = va_uncertainty_all > threshold_95
error_rate_out    = errors[outlier_mask].mean()
error_rate_norm   = errors[~outlier_mask].mean()
mean_va           = va_uncertainty_all.mean()
scaling_outliers  = (va_uncertainty_all[outlier_mask] / mean_va).mean()
print(f"\nError Rate Outliers (top 5% VA): {error_rate_out:.1%}")
print(f"Error Rate Normal:               {error_rate_norm:.1%}")
print(f"Avg Bound Expansion (Outliers):  {scaling_outliers:.1f}x wider")

Optimal Threshold (Youden): 0.3294
Error Prediction AUROC:     0.785
Mean VA (Errors):   0.0113
Mean VA (Correct):  0.0065
P-value:            8.40e-13

Error Rate Outliers (top 5% VA): 10.0%
Error Rate Normal:               7.7%
Avg Bound Expansion (Outliers):  9.8x wider


# Stage 2 Patient Selection
We select patients in two ways: By focusing on patients who actually need medication change (y_binary == 1) or by using the predicted probability of change (y_hat == 1)

In [7]:
if USE_PREDICTED_STAGE1:
    fpr, tpr, thresholds = roc_curve(y_test_stage1, p_test_uncalibrated)
    optimal_threshold = thresholds[np.argmax(tpr - fpr)]
    y_hat_stage1 = (p_all_uncalibrated > optimal_threshold).astype(int)
    mask = (y_hat_stage1 == 1)
    print(f"Selection: y_hat == 1 | Threshold: {optimal_threshold:.4f}")
else:
    mask = (y_binary == 1)
    print("Selection: y_binary == 1 (true positives)")

X_stage2             = X[mask]
y_stage2             = y_continuous[mask]
va_uncertainty_stage2 = va_uncertainty_all[mask]
p_lower_stage2       = p_lower_all[mask]
p_upper_stage2       = p_upper_all[mask]

print(f"Patients selected: {len(X_stage2)} / {len(X)} ({len(X_stage2)/len(X)*100:.1f}%)")

Selection: y_binary == 1 (true positives)
Patients selected: 1533 / 6105 (25.1%)


### Stage 2 Train/Test Split and Base Model

In [8]:
(X_train_2, X_test_2,
 y_train_2, y_test_2,
 va_unc_train_2, va_unc_test_2,
 p_lower_train_2, p_lower_test_2) = train_test_split(
    X_stage2, y_stage2, va_uncertainty_stage2, p_lower_stage2,
    test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train_2)} | Test: {len(X_test_2)}")

# Base model for MAPIE baselines
if STAGE1_MODEL_TYPE == "XGBoost":
    xgb_model_base = xgb.XGBRegressor(**XGBOOST_REGRESSION_PARAMS, random_state=RANDOM_STATE)
else:
    xgb_model_base = LinearRegression()
xgb_model_base.fit(X_train_2, y_train_2)

rmse = np.sqrt(mean_squared_error(y_test_2, xgb_model_base.predict(X_test_2)))
print(f"Base model RMSE: {rmse:.4f}")

# Proper train / calibration split (for CASCADE — reused across alphas)
X_train_proper, X_calib, y_train_proper, y_calib, va_unc_proper, va_unc_calib = train_test_split(
    X_train_2, y_train_2, va_unc_train_2, test_size=0.2, random_state=42
)
print(f"Proper Train: {len(X_train_proper)} | Calibration: {len(X_calib)}")

if STAGE1_MODEL_TYPE == "XGBoost":
    xgb_model_proper = xgb.XGBRegressor(**XGBOOST_REGRESSION_PARAMS, random_state=RANDOM_STATE)
else:
    xgb_model_proper = LinearRegression()
xgb_model_proper.fit(X_train_proper, y_train_proper)

Train: 1226 | Test: 307
Base model RMSE: 0.0995
Proper Train: 980 | Calibration: 246


XGBRegressor(alpha=0.1, base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, lambda=1, learning_rate=0.1,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=700, n_jobs=None, ...)

## Helper Functions

In [9]:
def calculate_metrics(y_true, y_lower, y_upper, uncertainties, k=3):
    """Coverage, Average Length, and Cascade Ratio (generalised for K bins)."""
    y_true = np.asarray(y_true).flatten()
    y_lower = np.asarray(y_lower).flatten()
    y_upper = np.asarray(y_upper).flatten()
    u_test  = np.asarray(uncertainties).flatten()

    covered  = (y_true >= y_lower) & (y_true <= y_upper)
    cov      = np.mean(covered)
    widths   = y_upper - y_lower
    avg_width = np.mean(widths)

    p_low  = 100.0 / k
    p_high = 100.0 * (k - 1) / k
    t_low  = np.percentile(va_unc_train_2, p_low)
    t_high = np.percentile(va_unc_train_2, p_high)

    mask_low  = u_test <= t_low
    mask_high = u_test >  t_high
    w_low  = np.mean(widths[mask_low])
    w_high = np.mean(widths[mask_high])
    ratio  = w_high / w_low if w_low > 1e-6 else 0.0

    return cov, avg_width, ratio

def get_scaling_factor_centered(va_uncertainty, global_mean_unc,
                                 beta=0.5, min_scale=0.2, max_scale=5.0):
    norm_diff = (va_uncertainty / global_mean_unc) - 1.0
    sigma     = 1.0 + (beta * norm_diff)
    return np.clip(sigma, a_min=min_scale, a_max=max_scale)

class CascadeEvaluator:
    def __init__(self, alpha):
        self.alpha = alpha
        
    def get_metrics(self, y_true, y_pred, y_lower, y_upper):
        y_true  = np.asarray(y_true).flatten()
        y_pred  = np.asarray(y_pred).flatten()
        y_low   = np.asarray(y_lower).flatten()
        y_high  = np.asarray(y_upper).flatten()
        widths  = y_high - y_low
        errors  = np.abs(y_true - y_pred)
        coverage = np.mean((y_true >= y_low) & (y_true <= y_high))
        winkler  = np.mean(widths + (2.0 / self.alpha) * (
            np.maximum(y_low - y_true, 0) + np.maximum(y_true - y_high, 0)
        ))
        corr, _ = stats.spearmanr(widths, errors)
        if np.isnan(corr): corr = 0.0
        return {"Winkler": float(winkler),
                "Adaptivity (Corr)": float(corr),
                "Coverage": float(coverage)}

def winkler_calc(y, low, high, alpha):
    w = high - low
    p = (2 / alpha) * (np.maximum(low - y, 0) + np.maximum(y - high, 0))
    return np.mean(w + p)

# Alpha Loop
Iterates over all alphas (0.3, 0.2, 0.1, 0.05) levels

In [11]:
all_results = {}  

y_true_arr = np.array(y_test_2).flatten()
unc_arr    = np.array(va_unc_test_2).flatten()
u_test     = unc_arr.copy()

global_mean_unc      = np.mean(va_unc_proper)
y_pred_calib_base    = xgb_model_proper.predict(X_calib)
residuals_calib_base = np.abs(y_calib - y_pred_calib_base)
y_pred_test_base     = xgb_model_proper.predict(X_test_2)

for ALPHA, cfg in ALPHAS_CONFIG.items():
    target_cov   = 1 - ALPHA
    beta_max     = cfg["beta_max"]
    beta_sel_cap = cfg["beta_sel_cap"]
    betas        = np.round(np.arange(0.0, beta_max + 0.05, 0.1), 2)

    print(f"  ALPHA = {ALPHA:.2f}  |  Target Coverage = {target_cov:.0%}  "
          f"|  Beta grid: 0.0 → {beta_max}")

    # Experiment 1: Baseline CP
    print("\nExperiment 1: Baseline Conformal Prediction")
    results_baseline = {}
    print(f"{'Strategy':<22} | {'Coverage':<10} | {'Avg Len':<10} | {'CR'}")
    print("-" * 60)
    for strategy, params in STRATEGIES.items():
        mapie = MapieRegressor(xgb_model_base, test_size=0.2,
                               **params, random_state=RANDOM_STATE)
        mapie.fit(X_train_2, y_train_2)
        y_pred, y_pis = mapie.predict(X_test_2, alpha=ALPHA)
        y_lower = y_pis[:, 0, 0]
        y_upper = y_pis[:, 1, 0]
        coverage, width, rat = calculate_metrics(y_true_arr, y_lower, y_upper,
                                           unc_arr, k=K_BINS)
        results_baseline[strategy] = {
            "y_pred": y_pred, "y_lower": y_lower, "y_upper": y_upper,
            "coverage": coverage, "width": width, "ratio": rat
        }
        print(f"{strategy:<22} | {coverage:<10.3f} | {width:<10.4f} | {rat:.2f}")

    # Experiment 2: Mondrian CP 
    print(f"\nExperiment 2: Mondrian (Stratified) CP  K={K_BINS}")
    percentiles = np.linspace(0, 100, K_BINS + 1)
    bin_edges   = np.percentile(va_unc_train_2, percentiles)
    bin_labels  = [str(i) for i in range(1, K_BINS + 1)]
    bins_train  = pd.cut(va_unc_train_2, bins=bin_edges, labels=bin_labels,
                          include_lowest=True, duplicates="drop")
    bins_test   = pd.cut(va_unc_test_2,  bins=bin_edges, labels=bin_labels,
                          include_lowest=True, duplicates="drop")

    results_stratified = {}
    for strategy, params in STRATEGIES.items():
        y_pred_all  = np.zeros(len(X_test_2))
        y_lower_all = np.zeros(len(X_test_2))
        y_upper_all = np.zeros(len(X_test_2))
        for cat in bin_labels:
            mask_train_c = (bins_train == cat)
            mask_test_c  = (bins_test  == cat)
            if mask_train_c.sum() < 10 or mask_test_c.sum() < 2:
                continue
            if STAGE1_MODEL_TYPE == "XGBoost":
                model_cat = xgb.XGBRegressor(**XGBOOST_REGRESSION_PARAMS,
                                              random_state=RANDOM_STATE)
            else:
                model_cat = LinearRegression()
            model_cat.fit(X_train_2[mask_train_c], y_train_2[mask_train_c])
            mapie_cat = MapieRegressor(model_cat, test_size=0.2,
                                        **params, random_state=RANDOM_STATE)
            mapie_cat.fit(X_train_2[mask_train_c], y_train_2[mask_train_c])
            y_pred_c, y_pis_c = mapie_cat.predict(X_test_2[mask_test_c], alpha=ALPHA)
            y_pred_all[mask_test_c]  = y_pred_c
            y_lower_all[mask_test_c] = y_pis_c[:, 0, 0]
            y_upper_all[mask_test_c] = y_pis_c[:, 1, 0]
        coverage, width, rat = calculate_metrics(y_true_arr, y_lower_all,
                                           y_upper_all, unc_arr, k=K_BINS)
        results_stratified[strategy] = {
            "y_pred": y_pred_all, "y_lower": y_lower_all, "y_upper": y_upper_all,
            "coverage": coverage, "width": width, "ratio": rat
        }
    strat_split = results_stratified["split"]
    print(f"  Mondrian (split) | Cov={strat_split['coverage']:.3f} "
          f"| Len={strat_split['width']:.4f} | CR={strat_split['ratio']:.2f}")

    # Experiment 3: Continuous CASCADE
    print(f"\nExperiment 3: Continuous CASCADE  |  beta grid: {betas[0]:.1f} → {betas[-1]:.1f}")
    print(f"{'Beta':<6} | {'Cov':>7} | {'Len':>7} | {'CR':>6} | {'p-val':>7} | Status")
    print("-" * 55)

    best_cov_error  = 1.0
    opt_beta        = 0.0
    best_ratio_opt  = 0.0
    best_len_opt    = float("inf")
    final_y_lower   = None
    final_y_upper   = None
    sensitivity_results = []

    for b in betas:
        sigma_calib = get_scaling_factor_centered(va_unc_calib, global_mean_unc, beta=b)
        scores      = residuals_calib_base / sigma_calib
        n_calib     = len(scores)
        q_level     = np.ceil((n_calib + 1) * target_cov) / n_calib
        q_hat       = np.quantile(scores, min(1.0, q_level), method="higher")

        sigma_test  = get_scaling_factor_centered(u_test, global_mean_unc, beta=b)
        half_width  = q_hat * sigma_test
        y_lower     = y_pred_test_base - half_width
        y_upper     = y_pred_test_base + half_width

        coverage, avg_len, ratio = calculate_metrics(y_true_arr, y_lower,
                                                  y_upper, u_test, k=K_BINS)
        
        # Bootstrap p-value vs Split CP baseline
        diffs = []
        for _ in range(N_BOOT):
            idx = resample(np.arange(len(y_true_arr)), replace=True)
            _, _, r_b    = calculate_metrics(y_true_arr[idx], y_lower[idx],
                                              y_upper[idx], u_test[idx], k=K_BINS)
            _, _, r_base = calculate_metrics(
                y_true_arr[idx],
                results_baseline["split"]["y_lower"][idx],
                results_baseline["split"]["y_upper"][idx],
                u_test[idx], k=K_BINS)
            diffs.append(r_b - r_base)
        p_val = np.mean(np.array(diffs) <= 0)

        status = "Valid"
        if coverage < target_cov:            status = "Invalid"
        elif coverage >= target_cov + 0.05:  status = "Conservative"

        sensitivity_results.append({
            "beta": b, "cov": coverage, "len": avg_len,
            "ratio": ratio, "p_val": p_val, "status": status
        })

        # Optimal beta selection: closest to nominal, then shortest length
        # Only consider betas up to beta_sel_cap
        if coverage >= target_cov and b <= beta_sel_cap:
            cov_error = abs(coverage - target_cov)
            if cov_error < best_cov_error - 1e-4:
                best_cov_error = cov_error
                best_len_opt   = avg_len
                opt_beta       = b
                best_ratio_opt = ratio
                final_y_lower  = y_lower.copy()
                final_y_upper  = y_upper.copy()
            elif abs(cov_error - best_cov_error) <= 1e-4 and avg_len < best_len_opt:
                best_len_opt   = avg_len
                opt_beta       = b
                best_ratio_opt = ratio
                final_y_lower  = y_lower.copy()
                final_y_upper  = y_upper.copy()

        p_str = "<.001" if p_val < 0.001 else f"{p_val:.3f}"
        print(f"{b:<6.1f} | {coverage:>6.2%} | {avg_len:>7.4f} | {ratio:>6.2f} | "
              f"{p_str:>7} | {status}")

    df_sens = pd.DataFrame(sensitivity_results)
    print(f"\n  → Optimal beta: {opt_beta:.1f}  CR: {best_ratio_opt:.2f}  "
          f"Len: {best_len_opt:.4f}")

    # Final CASCADE calibration 
    sigma_calib_opt = get_scaling_factor_centered(va_unc_calib, global_mean_unc, beta=opt_beta)
    sigma_test_opt  = get_scaling_factor_centered(va_unc_test_2, global_mean_unc, beta=opt_beta)
    scores_opt      = residuals_calib_base / sigma_calib_opt
    n_calib         = len(scores_opt)
    q_level         = np.ceil((n_calib + 1) * target_cov) / n_calib
    q_hat_opt       = np.quantile(scores_opt, min(1.0, q_level), method="higher")
    half_width_opt  = q_hat_opt * sigma_test_opt

    results_cascade = {
        "split": {
            "y_pred":  y_pred_test_base,
            "y_lower": y_pred_test_base - half_width_opt,
            "y_upper": y_pred_test_base + half_width_opt,
            "q_hat":   q_hat_opt,
        }
    }

    # Master table 
    evaluator   = CascadeEvaluator(alpha=ALPHA)
    master_rows = []
    for strat in ["split", "cv_plus", "jackknife_plus_ab"]:
        if strat in results_baseline:
            m = evaluator.get_metrics(
                y_test_2,
                results_baseline[strat]["y_pred"],
                results_baseline[strat]["y_lower"],
                results_baseline[strat]["y_upper"],
            )
            master_rows.append({"Method": f"Baseline CP ({strat})", **m})
    m_casc = evaluator.get_metrics(
        y_test_2,
        results_cascade["split"]["y_pred"],
        results_cascade["split"]["y_lower"],
        results_cascade["split"]["y_upper"],
    )
    master_rows.append({"Method": "Continuous CASCADE (Ours)", **m_casc})
    df_master = pd.DataFrame(master_rows)
    print(f"\nTABLE 1  (alpha={ALPHA:.2f})")
    print(df_master.to_string(index=False))

    all_results[ALPHA] = {
        "baseline":   results_baseline,
        "mondrian":   results_stratified,
        "cascade":    results_cascade,
        "opt_beta":   opt_beta,
        "df_sens":    df_sens,
    }

  ALPHA = 0.30  |  Target Coverage = 70%  |  Beta grid: 0.0 → 1.5

Experiment 1: Baseline Conformal Prediction
Strategy               | Coverage   | Avg Len    | CR
------------------------------------------------------------
split                  | 0.704      | 0.0595     | 1.00

Experiment 2: Mondrian (Stratified) CP  K=3
  Mondrian (split) | Cov=0.759 | Len=0.0580 | CR=1.39

Experiment 3: Continuous CASCADE  |  beta grid: 0.0 → 1.5
Beta   |     Cov |     Len |     CR |   p-val | Status
-------------------------------------------------------
0.0    | 67.10% |  0.0584 |   1.00 |   0.734 | Invalid
0.1    | 68.40% |  0.0619 |   1.22 |   <.001 | Invalid
0.2    | 68.73% |  0.0644 |   1.49 |   <.001 | Invalid
0.3    | 70.03% |  0.0681 |   1.81 |   <.001 | Valid
0.4    | 70.36% |  0.0732 |   2.21 |   <.001 | Valid
0.5    | 70.03% |  0.0739 |   2.71 |   <.001 | Valid
0.6    | 70.36% |  0.0787 |   3.36 |   <.001 | Valid
0.7    | 69.38% |  0.0855 |   4.23 |   <.001 | Invalid
0.8    | 68.40% |

## Alpha Ablation Table

In [16]:
print(f"{'Alpha':>6} | {'Target':>7} | {'Method':<25} | {'Coverage':>10} | {'Avg Len':>9} | {'CR':>6} | Opt β")
print("-" * 78)
for ALPHA, res in all_results.items():
    target_cov = 1 - ALPHA
    for name, dat in [
        ("Baseline (Split CP)",   res["baseline"]["split"]),
        ("Mondrian (Stratified)", res["mondrian"]["split"]),
        ("Continuous CASCADE",    res["cascade"]["split"]),
    ]:
        coverage, width, rat = calculate_metrics(
            y_true_arr,
            dat["y_lower"], dat["y_upper"],
            unc_arr, k=K_BINS
        )
        opt_b = f"{res['opt_beta']:.1f}" if "CASCADE" in name else "-"
        print(f"{ALPHA:>6.2f} | {target_cov:>7.0%} | {name:<25} | "
              f"{coverage:>10.3f} | {width:>9.3f} | {rat:>6.2f} | {opt_b}")
    print()

 Alpha |  Target | Method                    |   Coverage |   Avg Len |     CR | Opt β
------------------------------------------------------------------------------
  0.30 |     70% | Baseline (Split CP)       |      0.704 |     0.060 |   1.00 | -
  0.30 |     70% | Mondrian (Stratified)     |      0.759 |     0.058 |   1.39 | -
  0.30 |     70% | Continuous CASCADE        |      0.700 |     0.068 |   1.81 | 0.3

  0.20 |     80% | Baseline (Split CP)       |      0.840 |     0.113 |   1.00 | -
  0.20 |     80% | Mondrian (Stratified)     |      0.866 |     0.118 |   2.02 | -
  0.20 |     80% | Continuous CASCADE        |      0.801 |     0.148 |   4.23 | 0.7

  0.10 |     90% | Baseline (Split CP)       |      0.935 |     0.258 |   1.00 | -
  0.10 |     90% | Mondrian (Stratified)     |      0.961 |     0.376 |   2.47 | -
  0.10 |     90% | Continuous CASCADE        |      0.909 |     0.260 |   2.71 | 0.5

  0.05 |     95% | Baseline (Split CP)       |      0.977 |     0.492 |   1.00

## Additional: Cascade Ratio for Different K Values (per alpha)

In [13]:
k_list = [3, 5, 7]
u_t     = np.array(va_unc_test_2).flatten()
u_train = np.array(va_unc_train_2).flatten()

for ALPHA, res in all_results.items():
    print(f"\nAlpha = {ALPHA:.2f}  (Target {1-ALPHA:.0%})")
    appendix_rows = []
    casc_lower = res["cascade"]["split"]["y_lower"]
    casc_upper = res["cascade"]["split"]["y_upper"]
    y_t = np.array(y_test_2).flatten()

    for k in k_list:
        p_low  = 100.0 / k
        p_high = 100.0 * (k - 1) / k
        t_low  = np.percentile(u_train, p_low)
        t_high = np.percentile(u_train, p_high)
        mask_low  = u_t <= t_low
        mask_high = u_t >  t_high
        w_c    = casc_upper - casc_lower
        w_low  = np.mean(w_c[mask_low])
        w_high = np.mean(w_c[mask_high])
        cov_low  = np.mean((y_t[mask_low]  >= casc_lower[mask_low])  & (y_t[mask_low]  <= casc_upper[mask_low]))
        cov_high = np.mean((y_t[mask_high] >= casc_lower[mask_high]) & (y_t[mask_high] <= casc_upper[mask_high]))
        cr = w_high / w_low if w_low > 1e-6 else 0.0
        appendix_rows.append({
            "K": k, "Low Cov": f"{cov_low:.2%}", "Low Len": round(w_low, 4),
            "High Cov": f"{cov_high:.2%}", "High Len": round(w_high, 4),
            "Cascade Ratio": round(cr, 2)
        })
    print(pd.DataFrame(appendix_rows).to_string(index=False))


Alpha = 0.30  (Target 70%)
 K Low Cov  Low Len High Cov  High Len  Cascade Ratio
 3  63.11%   0.0529   76.04%    0.0958           1.81
 5  65.43%   0.0516   78.95%    0.1099           2.13
 7  65.67%   0.0510   78.05%    0.1175           2.30

Alpha = 0.20  (Target 80%)
 K Low Cov  Low Len High Cov  High Len  Cascade Ratio
 3  69.67%   0.0690   91.67%    0.2920           4.23
 5  70.37%   0.0624   92.98%    0.3651           5.85
 7  70.15%   0.0593   95.12%    0.4048           6.83

Alpha = 0.10  (Target 90%)
 K Low Cov  Low Len High Cov  High Len  Cascade Ratio
 3  86.07%   0.1621   95.83%    0.4389           2.71
 5  90.12%   0.1539   96.49%    0.5296           3.44
 7  89.55%   0.1501   97.56%    0.5789           3.86

Alpha = 0.05  (Target 95%)
 K Low Cov  Low Len High Cov  High Len  Cascade Ratio
 3  92.62%   0.2073   97.92%    0.8777           4.23
 5  95.06%   0.1875  100.00%    1.0975           5.85
 7  94.03%   0.1781  100.00%    1.2168           6.83


## Additional: Full Beta Ablation by Uncertainty Group (per alpha)

In [15]:
K_BINS_REPORT = 3
group_labels  = ["Low", "Medium", "High"] if K_BINS_REPORT == 3 else [f"Bin_{i+1}" for i in range(K_BINS_REPORT)]

u_t     = np.array(va_unc_test_2).flatten()
u_train = np.array(va_unc_train_2).flatten()
edges   = np.percentile(u_train, np.linspace(0, 100, K_BINS_REPORT + 1))
bins    = pd.cut(u_t, bins=edges, include_lowest=True, labels=group_labels)
y_t     = np.array(y_test_2).flatten()

for ALPHA, cfg in ALPHAS_CONFIG.items():
    beta_max      = cfg["beta_max"]
    betas_to_test = np.round(np.arange(0.0, beta_max + 0.05, 0.1), 2)
    target_cov    = 1 - ALPHA

    print(f"\nFull Beta Ablation  alpha={ALPHA:.2f}  (Target {target_cov:.0%})")
    table_ablation = []

    for b in betas_to_test:
        sigma_calib = get_scaling_factor_centered(np.array(va_unc_calib).flatten(),
                                                   global_mean_unc, beta=b)
        scores  = residuals_calib_base / sigma_calib
        n_calib = len(scores)
        q_level = np.ceil((n_calib + 1) * target_cov) / n_calib
        q_hat   = np.quantile(scores, min(1.0, q_level), method="higher")
        sigma_test = get_scaling_factor_centered(u_t, global_mean_unc, beta=b)
        casc_low  = y_pred_test_base - q_hat * sigma_test
        casc_high = y_pred_test_base + q_hat * sigma_test

        row = {"Beta": b}
        for group in group_labels:
            mask = (bins == group)
            if mask.sum() > 0:
                coverage     = np.mean((y_t[mask] >= casc_low[mask]) & (y_t[mask] <= casc_high[mask]))
                avg_len = np.mean(casc_high[mask] - casc_low[mask])
                row[f"{group} Cov"] = f"{coverage*100:.1f}%"
                row[f"{group} Len"] = f"{avg_len:.3f}"
        table_ablation.append(row)

    cols = ["Beta"] + [f"{g} Cov" for g in group_labels] + [f"{g} Len" for g in group_labels]
    cols = ["Beta"] + sum([[f"{g} Cov", f"{g} Len"] for g in group_labels], [])
    print(pd.DataFrame(table_ablation)[cols].to_string(index=False))


Full Beta Ablation  alpha=0.30  (Target 70%)
 Beta Low Cov Low Len Medium Cov Medium Len High Cov High Len
  0.0   63.9%   0.058      70.8%      0.058    67.7%    0.058
  0.1   63.9%   0.057      70.8%      0.059    71.9%    0.070
  0.2   63.1%   0.055      70.8%      0.059    74.0%    0.082
  0.3   63.1%   0.053      73.0%      0.059    76.0%    0.096
  0.4   62.3%   0.051      73.0%      0.060    78.1%    0.113
  0.5   60.7%   0.046      71.9%      0.057    80.2%    0.125
  0.6   59.8%   0.043      71.9%      0.057    82.3%    0.144
  0.7   54.9%   0.040      71.9%      0.058    85.4%    0.169
  0.8   50.8%   0.036      73.0%      0.060    86.5%    0.200
  0.9   50.0%   0.034      73.0%      0.061    88.5%    0.236
  1.0   50.8%   0.032      69.7%      0.061    90.6%    0.272
  1.1   48.4%   0.031      71.9%      0.063    91.7%    0.323
  1.2   48.4%   0.030      70.8%      0.060    92.7%    0.364
  1.3   48.4%   0.030      68.5%      0.054    92.7%    0.377
  1.4   48.4%   0.030   